# 1. MVT-тестирование
## Какие ключевые шаги необходимо выполнить для планирования и проведения MVT-теста?

MVT-тестирование — это метод экспериментального дизайна, при котором одновременно тестируются **несколько факторов** и их комбинации, чтобы оценить как индивидуальное влияние факторов, так и их **взаимодействия**.

Процесс проведения MVT-теста можно разделить на три основных этапа: подготовка, запуск и анализ результатов.

**Пример (сквозной кейс MVT)**

**Контекст:** интернет-магазин электроники

**Страница:** карточка товара смартфона

**Проблема:** низкая конверсия из просмотра карточки в покупку

**Цель:** увеличить конверсию в заказ за счёт изменений UI-элементов карточки товара

### Этап 1. Подготовка MVT-теста

1. **Формулировка бизнес-цели и гипотез**

* Определяется бизнес-цель эксперимента — проверить гипотезу о **совместном влиянии нескольких факторов** на ключевые метрики.
* Формулируются гипотезы как о **главных эффектах**, так и о **возможных взаимодействиях факторов**.
* В отличие от A/B-теста, MVT позволяет оценить не только лучший вариант, но и то, **какие элементы усиливают или ослабляют эффект друг друга**.

> **В рамках примера:**
>
> **Цель:** повысить конверсию в покупку на странице товара смартфона.
>
> **Гипотеза:**
> Совместное изменение заголовка карточки товара (A), основного изображения (B) и цвета кнопки «В корзину» (C) приведёт к росту конверсии в покупку по сравнению с текущей версией страницы.

2. **Выбор факторов и вариантов**

* Определяются ключевые элементы интерфейса, потенциально влияющие на поведение пользователя.
* Для каждого фактора задаётся ограниченное число вариантов, чтобы контролировать рост числа комбинаций.

> **В рамках примера:**
>
> **Фактор A — Заголовок товара**
>
> * A1: «Смартфон X — официальный магазин»
> * A2: «Смартфон X — доставка завтра, гарантия 2 года»
>
> **Фактор B — Основное изображение**
>
> * B1: стандартное фото товара
> * B2: lifestyle-изображение (смартфон в руке пользователя)
>
> **Фактор C — Цвет кнопки «В корзину»**
>
> * C1: серый
> * C2: зелёный

Полный факторный дизайн:
**2 × 2 × 2 = 8 комбинаций интерфейса**.

3. **Предварительный EDA**

Перед запуском MVT проводится **разведочный анализ исторических данных**, чтобы проверить корректность дизайна эксперимента.

* **Пропущенные значения**
   
   *Проблема:* отсутствуют значения чека или выручки для части пользователей.
   
   *Действие:* проверить долю пропусков по комбинациям; учитывать отсутствие покупки как 0, технические пропуски — исключать из анализа.

* **Экстремальные значения (выбросы)**

   *Проблема:* единичные пользователи с очень высоким чеком искажают средние значения.
   
   *Действие:* анализировать распределения, использовать медианы или винзоризацию, интерпретировать эффект для основной массы пользователей.

* **Ненормальные и скошенные распределения**

   *Проблема:* метрики выручки и AOV имеют длинный правый хвост.

   *Действие:* не использовать стандартный ANOVA; применять бутстреп, непараметрические тесты или GLM (например, Gamma-распределение для AOV).

* **Гетерогенность пользователей**
   
   *Проблема:* разные сегменты (устройства, новые/возвращающиеся) ведут себя по-разному.
   
   *Действие:* проверить метрики по ключевым когортам и учитывать их при интерпретации результатов.

> **В рамках примера:**
> Анализ исторических данных показал, что распределение AOV имеет выраженный длинный хвост, поэтому для оценки эффекта факторов на чек использовался бутстреп вместо ANOVA.

4. **Определение метрик**

* **Основные метрики:**

  * Конверсия в покупку (CR)
  * Средний чек (AOV)
  * Выручка на пользователя
* **Вторичные метрики:**

  * CTR кнопки «В корзину»
  * Время на странице
  * Bounce Rate
* Заранее фиксируется **OPE (Overall Evaluation Criterion)** — ключевая бизнес-метрика, по которой принимается итоговое решение.

> **В рамках примера:**
>
> **OPE:** общая выручка со страницы товара
> `(Выручка = Трафик × Конверсия × Средний чек)`

5. **Расчёт необходимого трафика и длительности**

* MVT требует существенно большего объёма данных, чем A/B-тест, из-за большого числа комбинаций.
* Минимальная длительность рассчитывается как:

$Минимальная длительность = \frac{(Минимальный\ размер\ группы × Количество\ комбинаций)}{ Ежедневный\ трафик}$

> **В рамках примера:**
>
> * Минимальный размер одной группы: 10 000 пользователей
> * Количество комбинаций: 8
> * Ежедневный трафик: 20 000 пользователей
>
> Минимальная длительность ≈ **4 дня**

При недостатке трафика применяется **дробный факторный дизайн (Fractional Factorial)**, позволяющий пожертвовать малозначимыми взаимодействиями.

---




### Этап 2. Запуск эксперимента

1. **Рандомизация и стратификация**

* Пользователи случайно распределяются между всеми комбинациями факторов.
* Доли трафика фиксированы.
* При необходимости применяется **стратификация**:
  пользователи сначала делятся на однородные когорты, после чего внутри каждой когорты выполняется рандомизация.

> **В рамках примера:**
>
> * Каждая комбинация получает ~12.5% трафика.
> * Стратификация проводится по признаку «новый / возвращающийся пользователь».

---

2. **Контроль корректности данных (EDA во время теста)**

В процессе MVT-теста проводится EDA для раннего выявления проблем с данными и дизайном эксперимента.

* **Дисбаланс групп**

  *Проблема:* одна комбинация получает больше мобильного трафика или новых пользователей.

  *Действие:* проверить распределение пользователей по устройствам, источникам и когортам; при сильном перекосе — признать эксперимент некорректным.

* **Выбросы и искажение метрик**

  *Проблема:* отдельные заказы с экстремально высоким чеком искажают AOV и выручку.

  *Действие:* мониторить распределения метрик; для анализа использовать робастные показатели и бутстреп.

* **Ненормальная форма распределений**

  *Проблема:* метрики имеют сильную асимметрию и длинные хвосты.

  *Действие:* отказаться от параметрических тестов в пользу непараметрических методов или GLM.

* **Внешние аномалии**

  *Проблема:* акции, сбои или внешние события влияют на поведение пользователей.

  *Действие:* зафиксировать событие и учитывать его при интерпретации или остановке теста.

> **В рамках примера:**
> в середине теста была запущена краткосрочная маркетинговая акция, что привело к всплеску трафика.
---

3. **Соблюдение дисциплины теста**

* Результаты не анализируются до завершения теста (без peeking).
* Длительность и критерии остановки зафиксированы заранее.
* Интерфейс не меняется в процессе эксперимента.

---


### Этап 3. Анализ результатов

1. **Оценка главных эффектов**

* Анализируется влияние каждого фактора отдельно:

  * эффект заголовка (A)
  * эффект изображения (B)
  * эффект цвета кнопки (C)

> **В рамках примера:**
> Цвет кнопки C2 показал наибольший вклад в рост конверсии.

2. **Анализ взаимодействий факторов**

* Проверяется, усиливают ли факторы эффект друг друга или работают только в определённых сочетаниях.

> **В рамках примера:**
> Кнопка C2 значительно повышает конверсию **только в сочетании с изображением B2**, тогда как с B1 эффект минимален.

3. **Выбор статистического метода**

* ANOVA — при нормальных данных.
* GLM, бутстреп, непараметрические тесты — при скошенных распределениях.
* Анализ проводится как по всей выборке, так и по ключевым когортам.

> **В рамках примера:**
> Для анализа AOV используется бутстреп, так как распределение среднего чека ненормально.

4. **Бизнес-интерпретация и принятие решения**

* Рассчитывается влияние победившей комбинации на OPE.
* Выполняется сценарный анализ.
* Принимается решение: rollout, доработка или повторный тест.

> **В рамках примера:**
> Комбинация **A2 + B2 + C2** увеличила выручку на **7%**, что делает её кандидатом на внедрение.

---

## 2.1. В чем основное отличие MVT от A/B-тестирования?

**A/B-тестирование** представляет собой экспериментальный метод, направленный на сравнение двух вариантов одного фактора с целью оценки его изолированного влияния на целевую метрику.

**MVT-тестирование (Multivariate Testing)** предполагает одновременное варьирование нескольких факторов и позволяет оценивать как **главные эффекты отдельных факторов**, так и **эффекты их взаимодействия**.

Ключевое отличие MVT от A/B-тестирования заключается в том, что MVT обеспечивает более глубокое понимание структуры влияния интерфейсных элементов, позволяя выявлять не только наиболее эффективные варианты, но и **зависимость эффекта одного фактора от уровней других факторов**, что невозможно при классическом A/B-подходе.

## 2.2. Какие метрики обычно используются для оценки эффективности MVT?

Для количественной оценки влияния факторов и их комбинаций в MVT-тестах применяются как **ключевые бизнес-метрики**, так и **поведенческие показатели пользователей**.

**Основные метрики:**

* **Конверсия (CR)** — доля пользователей, совершивших целевое действие (покупку, подписку и т.д.).
* **Средний или медианный чек (AOV / Median Order Value)** — финансовая эффективность изменений; медиана предпочтительна при скошенных распределениях и наличии выбросов.
* **Выручка на пользователя** — позволяет оценить влияние комбинаций факторов на доход.

**Поведенческие метрики:**

* **CTR кнопки / элементов интерфейса** — кликабельность ключевых элементов.
* **Bounce Rate** — доля пользователей, покинувших страницу без взаимодействия.
* **Время на странице** — индикатор вовлечённости.

**Ключевой показатель для принятия решения:**

* **OPE (Overall Evaluation Criterion)** — агрегированная метрика, отражающая бизнес-ценность изменений. Именно на основе OPE принимается решение о внедрении или корректировке вариантов.

## 2.3. Как определить количество групп в MVT?

Количество экспериментальных групп в MVT-тесте определяется как **произведение числа вариантов всех факторов**:

$N_{\text{групп}} = n_1 \times n_2 \times \dots \times n_k$,

где ($n_k$) — количество уровней ($k$)-го фактора.

**Пример:**
Если тестируются 3 фактора по 2 уровня, общее число комбинаций равно:

$2 \times 2 \times 2 = 8 \text{ групп}$


При ограниченном объёме трафика рекомендуется использование **Fractional Factorial Design**, которое позволяет сократить количество тестируемых комбинаций за счёт исключения малозначимых взаимодействий между факторами.

## 2.4. Какие риски связаны с проведением MVT?

Проведение MVT-тестов связано с рядом методологических и практических рисков, которые необходимо учитывать на этапе планирования:

* **Недостаточный объём данных** — малое число участников в каждой комбинации снижает статистическую мощность теста и увеличивает вероятность ошибки второго рода (необнаружение реального эффекта).
* **Сложность интерпретации взаимодействий** — с увеличением числа факторов возрастает сложность анализа их совместного влияния на ключевые метрики; отдельные эффекты могут маскировать друг друга.
* **Повышенный риск ложных выводов при множественных сравнениях** — большое число комбинаций увеличивает вероятность ошибок первого рода (ложноположительных результатов), что требует применения корректировок для множественных тестов (например, Bonferroni, Holm).
* **Высокие требования к трафику и инфраструктуре** — каждая дополнительная комбинация требует отдельной группы пользователей, что увеличивает нагрузку на систему, сбор данных и аналитическую обработку.
* **Влияние внешних факторов** — маркетинговые акции, технические сбои или сезонные колебания трафика могут искажать результаты теста и требовать дополнительных корректировок или стратификации.

## 2.5. Как интерпретировать взаимодействие факторов в MVT?

Взаимодействие факторов (interaction effects) возникает, когда влияние одного элемента интерфейса на ключевую метрику зависит от значений другого элемента.

* **Пример:** цвет кнопки сам по себе не оказывает значимого влияния на конверсию, однако в комбинации с определённым заголовком наблюдается существенный рост конверсии.

Анализ взаимодействий факторов выполняется с помощью:

* статистических моделей с учётом взаимодействий (ANOVA с interaction terms, GLM);
* сравнения метрик для различных комбинаций факторов;
* визуализации эффектов (например, тепловые карты или графики взаимодействий), позволяющей наглядно оценить, какие сочетания элементов наиболее эффективны.

Такой подход позволяет не только выявить лучший вариант интерфейса, но и понять, **почему и в каких комбинациях элементы усиливают или ослабляют эффект друг друга**, что критично для оптимизации пользовательского опыта и повышения бизнес-метрик.


## **Вывод**

MVT-тестирование — мощный инструмент оптимизации пользовательского опыта, позволяющий выявлять сложные зависимости между элементами интерфейса. Однако его успешное применение требует тщательного планирования, достаточного трафика и корректной статистической интерпретации результатов.